- Esse notebook servirá pra grantir que o PCA, o t-SNE e o UMAP vão consumir exatamente os mesmos dados, consolidando toda a engenharia de "Ataque vs. Defesa" e o status do Super Bowl que foi feito nas etapas anteriores de EDA, regras de associação, e outliers.

In [4]:
import pandas as pd
import nflreadpy as nfl

# importação dos dados

ANO_ATUAL = 2026
ultimos_5_anos = list(range(ANO_ATUAL - 5, ANO_ATUAL))

df_jogos = nfl.load_schedules(ultimos_5_anos).to_pandas()
df_pstats = nfl.load_player_stats(ultimos_5_anos).to_pandas()

df_reg = df_jogos[df_jogos['game_type'] == 'REG'].copy()
df_sb = df_jogos[df_jogos['game_type'] == 'SB'].copy()

# agregando jogo a jogo

df_pstats['turnovers_cometidos'] = df_pstats['passing_interceptions'] + df_pstats['fumbles_lost_total']

df_game_team = df_pstats.groupby(['season', 'game_id', 'team']).agg(
    Pass_Yds=('passing_yards', 'sum'),
    Rush_Yds=('rushing_yards', 'sum'),
    Carries=('carries', 'sum'),
    Turnovers=('turnovers_cometidos', 'sum'),
    Sacks_Suffered=('sacks_suffered', 'sum')
).reset_index()

# cruzando mandante e visitante

df_games = df_reg[['season', 'game_id', 'home_team', 'away_team', 'home_score', 'away_score']].copy()

df_games = df_games.merge(df_game_team, left_on=['game_id', 'home_team'], right_on=['game_id', 'team'], how='left')
df_games.rename(columns={'Pass_Yds':'home_pass', 'Rush_Yds':'home_rush', 'Carries':'home_carries', 
                         'Turnovers':'home_to', 'Sacks_Suffered':'home_sacks'}, inplace=True)
df_games.drop(columns=['team'], inplace=True, errors='ignore')

df_games = df_games.merge(df_game_team, left_on=['game_id', 'away_team'], right_on=['game_id', 'team'], how='left')
df_games.rename(columns={'Pass_Yds':'away_pass', 'Rush_Yds':'away_rush', 'Carries':'away_carries', 
                         'Turnovers':'away_to', 'Sacks_Suffered':'away_sacks'}, inplace=True)
df_games.drop(columns=['team'], inplace=True, errors='ignore')

df_games.fillna(0, inplace=True)


# consolidando a temporada (Ataque vs. Defesa)

df_home_persp = pd.DataFrame({
    'season': df_games['season'], 'team': df_games['home_team'],
    'PF': df_games['home_score'], 'PA': df_games['away_score'],
    'Off_Pass_Yds': df_games['home_pass'], 'Off_Rush_Yds': df_games['home_rush'],
    'Off_Carries': df_games['home_carries'], 'Off_Turnovers': df_games['home_to'],
    'Def_Sacks_Produced': df_games['away_sacks'], 
    'Def_Turnovers_Forced': df_games['away_to'],
    'Vitoria': (df_games['home_score'] > df_games['away_score']).astype(int)
})

df_away_persp = pd.DataFrame({
    'season': df_games['season'], 'team': df_games['away_team'],
    'PF': df_games['away_score'], 'PA': df_games['home_score'],
    'Off_Pass_Yds': df_games['away_pass'], 'Off_Rush_Yds': df_games['away_rush'],
    'Off_Carries': df_games['away_carries'], 'Off_Turnovers': df_games['away_to'],
    'Def_Sacks_Produced': df_games['home_sacks'],
    'Def_Turnovers_Forced': df_games['home_to'],
    'Vitoria': (df_games['away_score'] > df_games['home_score']).astype(int)
})

df_season = pd.concat([df_home_persp, df_away_persp]).groupby(['season', 'team']).sum().reset_index()
df_season.rename(columns={'Vitoria': 'Reg_Season_Wins'}, inplace=True)


# mapeamento do superbowl

def status_superbowl(row):
    sb_ano = df_sb[df_sb['season'] == row['season']]
    if sb_ano.empty: return '-' 
    home, away = sb_ano.iloc[0]['home_team'], sb_ano.iloc[0]['away_team']
    vencedor = home if sb_ano.iloc[0]['home_score'] > sb_ano.iloc[0]['away_score'] else away
    if row['team'] == vencedor: return 'Campeão'
    elif row['team'] in [home, away]: return 'Vice'
    else: return 'Não Chegou'

df_season['Status_SB'] = df_season.apply(status_superbowl, axis=1)

In [5]:
display(df_season)
print(df_season.columns)

,season,team,PF,PA,Off_Pass_Yds,Off_Rush_Yds,Off_Carries,Off_Turnovers,Def_Sacks_Produced,Def_Turnovers_Forced,Reg_Season_Wins,Status_SB
0,2021,ARI,449,366,4619,2076,496,15,41,27,11,Não Chegou
1,2021,ATL,313,459,3987,1451,393,26,18,20,7,Não Chegou
2,2021,BAL,387,392,4267,2479,517,26,34,15,8,Não Chegou
3,2021,BUF,483,289,4450,2209,461,22,42,30,11,Não Chegou
4,2021,CAR,304,404,3573,1842,455,29,39,16,5,Não Chegou
...,...,...,...,...,...,...,...,...,...,...,...,...
155,2025,SEA,483,292,4063,2096,507,28,47,25,14,Campeão
156,2025,SF,437,371,4318,1817,481,22,20,16,12,Não Chegou
157,2025,TB,380,411,3755,1947,472,16,37,23,8,Não Chegou
158,2025,TEN,284,478,3241,1589,378,19,42,14,3,Não Chegou


Index(['season', 'team', 'PF', 'PA', 'Off_Pass_Yds', 'Off_Rush_Yds',
       'Off_Carries', 'Off_Turnovers', 'Def_Sacks_Produced',
       'Def_Turnovers_Forced', 'Reg_Season_Wins', 'Status_SB'],
      dtype='str')


In [6]:
# exportando para Parquet (Fast, Typeless, Compact)
caminho_arquivo = "dados_reducao.parquet"
df_season.to_parquet(caminho_arquivo, index=False)

print(f"✅ ETL Concluído! Base consolidada com {df_season.shape[0]} times/temporadas.")
print(f"💾 Arquivo salvo com sucesso em: {caminho_arquivo}")

✅ ETL Concluído! Base consolidada com 160 times/temporadas.
💾 Arquivo salvo com sucesso em: dados_reducao.parquet
